# Day 12 — Feature Engineering & Feature Selection

In [5]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', 50)

## Task 1 — من الأعمدة إلى الـ Features
(مثال من مجال كرة القدم: توقّع فوز الفريق — غيّره لمجالك إذا تبي)

**1) الفرق بين Feature Engineering و Feature Selection**
- **Feature Engineering**: *أصنع* أعمدة جديدة من الموجودة. يزيد المعلومات اللي يشوفها الموديل.
- **Feature Selection**: *أختار* أي الأعمدة تدخل الموديل وأحذف الباقي. يقلل الضجيج والتكرار.

**Target:** هل فاز الفريق؟ (1 = فوز، 0 = غير ذلك)

**Features يمكن توليدها:**
- `Goal_Diff_Last5` = متوسط فارق الأهداف في آخر 5 مباريات
- `Is_Home` = 1 إذا لعب على ملعبه
- `Rest_Days` = الأيام بين المباراة السابقة والحالية

**Features أحذفها:**
- `Match_ID` (معرّف، ما يتنبأ بشيء)
- `Stadium_Name` (يتكرر مع `Is_Home` وفيه قيم كثيرة)
- `Final_Score` (Data Leakage: النتيجة النهائية تكشف الجواب)

**2) جدول عائلات الاختيار**

| العائلة | الفكرة | مثال | ميزة | عيب |
|---|---|---|---|---|
| Filter | أقيّم كل عمود بالإحصاء قبل أي موديل | `corr()`، نسبة الموافقة لكل فئة | سريع وبسيط | ما يشوف التفاعل بين الأعمدة |
| Wrapper | أجرّب مجموعات أعمدة بموديل حقيقي وأقارن الأداء | RFE | أدق لأنه يستخدم الموديل | بطيء ومكلف |
| Embedded | الموديل نفسه يختار أثناء التدريب | Lasso، أهمية الأعمدة في Random Forest | توازن بين السرعة والدقة | مرتبط بموديل معيّن |

**لماذا Filter فقط هذا الأسبوع؟** لأننا ما دربنا موديل بعد؛ Filter ما يحتاج موديل، سريع، ونفهم الداتا بالأرقام قبل ما نبني أي شي.

**3) Data Leakage**
- `Disbursement_Date` (تاريخ صرف القرض): **لا يجوز**. يصير فقط بعد الموافقة، فهو يكشف الجواب.
- `Manager_Final_Decision` (قرار المدير النهائي): **لا يجوز**. هو الـ Target نفسه تقريبًا.
- `Customer_Since_Years` (سنوات التعامل): **يجوز**. معلومة موجودة قبل القرار.

**ماذا يحدث لو استخدمناهما؟** الموديل يعطي دقة شبه 100% في التدريب، لكنه يفشل في الواقع لأن هذي المعلومات ما تكون متوفرة وقت التنبؤ.

**4) «ارتباط قريب من الصفر يعني العمود لا يفيد» — هل صحيح؟**
لا. `corr()` يقيس العلاقة **الخطية** فقط. مثال من حياتي: النوم وأدائي في الدراسة؛ قليل النوم يضرني وكثيره يضرني، فالعلاقة على شكل قوس (U) والارتباط الخطي قريب من الصفر، مع أن النوم مهم جدًا.

## Task 2 — ولّد Features تعبّر عن القدرة على السداد

### 1) تحميل الملف

In [6]:
df = pd.read_csv('loan_clean.csv')
TARGET = 'Target'   # غيّر الاسم إذا عمود الهدف عندك مختلف
print(df.shape)
df.head()

(500, 9)


,Loan_ID,Gender,Married,Education,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Target
0,SIM0000,Female,No,Graduate,4900,708,171,180,1
1,SIM0001,Male,No,Not Graduate,7870,3153,232,300,1
2,SIM0002,Male,No,Not Graduate,6862,3155,240,120,0
3,SIM0003,Female,Yes,Graduate,6808,372,192,360,1
4,SIM0004,Male,Yes,Graduate,10183,0,284,360,1


### 2) إنشاء الأعمدة الجديدة
`LoanAmount` بالآلاف، لذلك نضربه في 1000 ليصير بالريال.

In [7]:
df['Total_Income']     = df['ApplicantIncome'] + df['CoapplicantIncome']
df['Has_Coapplicant']  = (df['CoapplicantIncome'] > 0).astype(int)
df['Loan_to_Income']   = (df['LoanAmount'] * 1000) / df['Total_Income']
df['Monthly_Payment']  = (df['LoanAmount'] * 1000) / df['Loan_Amount_Term']
df['Payment_to_Income']= df['Monthly_Payment'] / df['Total_Income']

### 3) أول 5 صفوف و describe()

In [8]:
new_cols = ['Loan_to_Income', 'Payment_to_Income']
display(df[new_cols].head())
display(df[new_cols].describe())

,Loan_to_Income,Payment_to_Income
0,30.492154,0.169401
1,21.046902,0.070156
2,23.959269,0.199661
3,26.740947,0.074280
4,27.889620,0.077471


,Loan_to_Income,Payment_to_Income
count,500.000000,500.000000
mean,32.415549,0.114998
std,11.177129,0.062929
min,6.387509,0.017743
25%,25.074148,0.076843
50%,30.633340,0.096056
75%,37.892163,0.136734
max,96.674401,0.703582


### 4) تقسيم إلى 4 فئات بـ pd.qcut وحساب نسبة الموافقة

In [9]:
labels = ['Q1_low', 'Q2', 'Q3', 'Q4_high']
df['Payment_to_Income_bin'] = pd.qcut(df['Payment_to_Income'], 4, labels=labels)
df['ApplicantIncome_bin']   = pd.qcut(df['ApplicantIncome'], 4, labels=labels)

for col in ['Payment_to_Income_bin', 'ApplicantIncome_bin', 'Has_Coapplicant']:
    print(df.groupby(col, observed=True)[TARGET].mean().round(3), '\n')

Payment_to_Income_bin
Q1_low     0.584
Q2         0.592
Q3         0.536
Q4_high    0.432
Name: Target, dtype: float64 

ApplicantIncome_bin
Q1_low     0.584
Q2         0.464
Q3         0.576
Q4_high    0.520
Name: Target, dtype: float64 

Has_Coapplicant
0    0.541
1    0.530
Name: Target, dtype: float64 



### 5) ملاحظات
- **غيّرت النسبة بين فئاتها:** `Payment_to_Income` — (اكتب هنا اتجاه التغيّر الذي ظهر عندك: هل ترتفع الموافقة أو تنخفض من Q1 إلى Q4؟)
- **لم يغيّر شيئًا تقريبًا:** (اكتب العمود الذي تقاربت نسب فئاته، غالبًا `ApplicantIncome` أو `Has_Coapplicant`)
- **هل النتيجة كما توقعت؟** (نعم/لا مع السبب. توقعي كان أن الدفعة العالية نسبةً للدخل تقلل الموافقة، لأن البنك يهمه القدرة على السداد لا الدخل وحده.)
*(عدّل الجمل أعلاه حسب الأرقام التي تظهر عندك فعلًا.)*

### 6) حفظ الملف

In [10]:
df.to_csv('loan_features.csv', index=False)

## Task 3 — اختر الـ Features بطرق Filter

### 1) المعرّفات والأعمدة الثابتة

In [11]:
print(df.nunique())
ids      = [c for c in df.columns if df[c].nunique() == len(df)]
constant = [c for c in df.columns if df[c].nunique() == 1]
print('Identifiers:', ids)
print('Constant   :', constant)

Loan_ID                  500
Gender                     2
Married                    2
Education                  2
ApplicantIncome          475
CoapplicantIncome        222
LoanAmount               211
Loan_Amount_Term           5
Target                     2
Total_Income             485
Has_Coapplicant            2
Loan_to_Income           500
Monthly_Payment          289
Payment_to_Income        500
Payment_to_Income_bin      4
ApplicantIncome_bin        4
dtype: int64
Identifiers: ['Loan_ID', 'Loan_to_Income', 'Payment_to_Income']
Constant   : []


### 2) الأعمدة التصنيفية: نسبة الموافقة لكل مجموعة ثم الفجوة

In [12]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.drop(ids, errors='ignore')
rows = []
for col in cat_cols:
    rates = df.groupby(col, observed=True)[TARGET].mean()
    rows.append({'column': col, 'max_rate': rates.max(), 'min_rate': rates.min(),
                 'gap': rates.max() - rates.min()})
gap_df = pd.DataFrame(rows).sort_values('gap', ascending=False)
gap_df

,column,max_rate,min_rate,gap
3,Payment_to_Income_bin,0.592000,0.432000,0.160000
4,ApplicantIncome_bin,0.584000,0.464000,0.120000
2,Education,0.595745,0.512535,0.083210
1,Married,0.551181,0.520325,0.030856
0,Gender,0.547619,0.533654,0.013965


### 3) الأعمدة الرقمية: corr() مع الـ Target

In [13]:
num_cols = df.select_dtypes(include='number').columns
corr_target = df[num_cols].corr()[TARGET].drop(TARGET)
corr_target.reindex(corr_target.abs().sort_values(ascending=False).index)

,Target
Monthly_Payment,-0.111928
Payment_to_Income,-0.110471
LoanAmount,-0.100383
Loan_Amount_Term,0.081114
Loan_to_Income,-0.077742
Total_Income,-0.028302
ApplicantIncome,-0.022641
CoapplicantIncome,-0.017334
Has_Coapplicant,-0.010300


### 4) التكرار بين الأعمدة

In [14]:
print(df[['ApplicantIncome', 'Total_Income']].corr())

                 ApplicantIncome  Total_Income
ApplicantIncome         1.000000      0.842211
Total_Income            0.842211      1.000000


إذا كان الارتباط عاليًا جدًا (مثلًا > 0.9) فالعمودان يحملان نفس المعلومة تقريبًا، فلا نحتاج الاثنين. أبقي `Total_Income` (لأنه أقرب لقدرة العميل على السداد) وأحذف `ApplicantIncome`.

### 5) جدول القرار

| العمود | القرار | الدليل بالرقم |
|---|---|---|
| Loan_ID (معرّف) | احذف | `nunique` = عدد الصفوف |
| أي عمود ثابت | احذف | `nunique` = 1 |
| Payment_to_Income | أبقِ | فرق نسبة الموافقة بين Q1 و Q4 كبير (اكتب الرقم) |
| Total_Income | أبقِ | ارتباط مع الـ Target (اكتب الرقم)، ونحذف ApplicantIncome لتكراره |
| ApplicantIncome | احذف | مكرر مع Total_Income (corr = اكتب الرقم) |
| Gender | **انتبه** | انظر الفقرة التالية |

**ماذا تقول الأرقام عن Gender؟** انظر الفجوة في جدول الخطوة 2. حتى لو ظهرت فجوة، هذا لا يعني أن الجنس سبب الموافقة؛ قد يكون سببها الدخل أو نوع الوظيفة. قراري: **أحذف Gender** حتى لو أعطى فجوة، لأن استخدامه يعرّض البنك لقرارات غير عادلة وغير قانونية، وأي تحسّن في الدقة منه لا يستحق هذا الخطر.

**ماذا قلت يوم الأربعاء عن العدل؟** (اكتب هنا ما قلته في Notebook الأربعاء، وقارنه بالأرقام اليوم.)
*(املأ الأرقام في الجدول بعد تشغيل الخلايا.)*

### 6) حفظ model_ready.csv

In [15]:
# عدّل القائمة حسب جدول قرارك
keep = ['Total_Income', 'Has_Coapplicant', 'Loan_to_Income', 'Payment_to_Income',
        'LoanAmount', 'Loan_Amount_Term']   # + أي أعمدة تصنيفية قررت إبقاءها
keep = [c for c in keep if c in df.columns]
model_ready = df[keep + [TARGET]]
model_ready.to_csv('model_ready.csv', index=False)
print(model_ready.shape)

(500, 7)


## Task 5 — حساسات مسبار الحياة

In [16]:
rng = np.random.default_rng(2026)
n = 300
life = rng.integers(0, 2, n)   # 1 = signs of life found
probes = pd.DataFrame({
    "probe_id": [f"P{i:03d}" for i in range(n)],
    "mission_code": "AX-7",
    "methane_ppm": np.round(5 + life * 12 + rng.normal(0, 3, n), 1),
    "temperature_c": np.round(-40 + life * 25 + rng.normal(0, 15, n), 1),
    "radiation": np.round(rng.normal(100, 10, n), 1),
    "humidity": np.round(20 + life * 6 + rng.normal(0, 8, n), 1),
    "life_detected": life,
})
probes["temperature_f"] = probes["temperature_c"] * 9 / 5 + 32

### 1) قواعد Filter: المعرّف والثابت

In [17]:
print(probes.nunique())
# probe_id: nunique = 300 (معرّف) | mission_code: nunique = 1 (ثابت)

probe_id         300
mission_code       1
methane_ppm      160
temperature_c    246
radiation        212
humidity         192
life_detected      2
temperature_f    246
dtype: int64


### 2) الارتباط مع الـ Target مرتبًا

In [18]:
sensors = ['methane_ppm', 'temperature_c', 'temperature_f', 'radiation', 'humidity']
corr = probes[sensors + ['life_detected']].corr()['life_detected'].drop('life_detected')
corr.reindex(corr.abs().sort_values(ascending=False).index)

,life_detected
methane_ppm,0.882036
temperature_f,0.643020
temperature_c,0.643020
humidity,0.397488
radiation,0.109774


### 3) حساسان يقيسان الشيء نفسه

In [19]:
print(probes[['temperature_c', 'temperature_f']].corr())

               temperature_c  temperature_f
temperature_c            1.0            1.0
temperature_f            1.0            1.0


`temperature_c` و `temperature_f` هما نفس القياس بوحدتين مختلفتين (F = C × 9/5 + 32)، والارتباط بينهما **1.0**. **أحذف `temperature_f`** وأبقي `temperature_c`، لأن الاثنين يحملان معلومة واحدة تمامًا.

### 4) متوسط كل حساس حسب وجود الحياة

In [20]:
probes.groupby('life_detected')[['methane_ppm', 'temperature_c', 'radiation', 'humidity']].mean().round(2)

,methane_ppm,temperature_c,radiation,humidity
life_detected,,,,
0,5.38,-40.31,98.74,19.14
1,17.04,-14.78,100.96,25.63


### 5) القرار

الحساسات الثلاثة مرتبة حسب الأهمية:

| الترتيب | الحساس | الدليل |
|---|---|---|
| 1 | `methane_ppm` | أعلى ارتباط: **0.88**. المتوسط 17.04 مع الحياة مقابل 5.38 بدونها |
| 2 | `temperature_c` | ارتباط **0.64**. المتوسط -14.78 مقابل -40.31 |
| 3 | `humidity` | ارتباط **0.40**. المتوسط 25.63 مقابل 19.14 |

**المستبعدات:**
- `temperature_f`: مكرر 100% مع `temperature_c`.
- `radiation`: ارتباط **0.11** فقط، والمتوسطان متقاربان (100.96 مقابل 98.74)، فهو أقرب للضجيج.
- `probe_id`: معرّف (300 قيمة مختلفة من 300).
- `mission_code`: ثابت (قيمة واحدة).

*(الأرقام من تشغيل الكود بنفس الـ seed = 2026؛ تأكد أنها تطابق مخرجاتك.)*

### 6) تفكير
لو عندنا وقت لتجربة موديل حقيقي أستخدم **Wrapper** (مثل RFE): أدرّب موديلًا على كل مجموعة من الحساسات وأقارن الأداء بـ Cross-Validation، وأتأكد أن المجموعة الثلاثية المختارة فعلًا هي الأفضل. السبب أن Filter يقيّم كل حساس لوحده، بينما Wrapper يختبر تفاعل الحساسات معًا. (و Embedded مثل أهمية الأعمدة في Random Forest خيار ثانٍ جيد وأسرع.)